# Math Basics - Ames Housing Dataset

## Task 1: Compute Mean and Standard Deviation Manually

## Task 2: Standardize One Column Using Broadcasting

## Task 3: Compute Cosine Similarity

## Task 4: Estimate Probability

In [14]:
import kagglehub
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from cleaning_01 import clean_data


path = kagglehub.dataset_download("prevek18/ames-housing-dataset")
print("Path to dataset files:", path)

df = pd.read_csv(os.path.join(path, "AmesHousing.csv"))

df.head()

Using Colab cache for faster access to the 'ames-housing-dataset' dataset.
Path to dataset files: /kaggle/input/ames-housing-dataset


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


---
## Task 1: Compute Mean and Standard Deviation Manually (Using NumPy)

In [15]:
def calculate_stats(df, target_column):
    target = df[target_column].values

    n = len(target)

    # Manual Mean
    manual_mean = np.sum(target) / n

    # Manual Std
    variance = np.sum((target - manual_mean) ** 2) / n
    manual_std = np.sqrt(variance)

    # Print results
    print(f"Manual Mean of {target_column}: {manual_mean}")
    print(f"Manual Std of {target_column}: {manual_std}")

    # Verification
    print(f"Verification with pandas .mean(): {df[target_column].mean()}")
    print(f"Verification with pandas .std(): {df[target_column].std()}")

    return manual_mean, manual_std

---
## Task 2: Standardize One Column Using Broadcasting

In [16]:
def standardize_feature(df, feature_column):
    X = df[feature_column].values

    # Manual standardization
    X_mean = np.mean(X)
    X_std = np.std(X)
    z_manual = (X - X_mean) / X_std

    # Using StandardScaler
    scaler = StandardScaler()
    z_scaler = scaler.fit_transform(df[[feature_column]]).reshape(-1)
    # Print results
    print(f"Manual Standardization (first 5 values): {z_manual[:5]}")
    print(f"StandardScaler (first 5 values): {z_scaler[:5]}")
    print(f"Are they equal? {np.allclose(z_manual, z_scaler)}")

    return z_manual, z_scaler

---
## Task 3: Compute Cosine Similarity

Find the highest-value and lowest-value records and compute cosine similarity between their feature vectors.

In [17]:
def analyze_similarity(df, numeric_cols, target_column='SalePrice'):

    # Clean data
    data_subset = df[numeric_cols].dropna()
    df_temp = df.loc[data_subset.index].copy()

    # Get highest & lowest indices
    highest_idx = df_temp[target_column].idxmax()
    lowest_idx = df_temp[target_column].idxmin()

    # Print values
    print(f"Highest value record (index {highest_idx}):")
    print(f"  {target_column}: ${df.loc[highest_idx, target_column]:,}")

    print(f"Lowest value record (index {lowest_idx}):")
    print(f"  {target_column}: ${df.loc[lowest_idx, target_column]:,}")

    # Extract vectors
    vector_high = data_subset.loc[highest_idx].values.reshape(1, -1)
    vector_low = data_subset.loc[lowest_idx].values.reshape(1, -1)

    # Cosine similarity
    cos_sim = cosine_similarity(vector_high, vector_low)[0][0]

    print(f"Cosine Similarity: {cos_sim:.4f}")

    return highest_idx, lowest_idx, cos_sim

---
## Task 4: Estimate Probability

What fraction of high-quality items have a target above a certain threshold?

In [18]:
def analyze_quality_price(df, quality_threshold=7, price_threshold=200000):

    # Filter high quality
    high_quality = df[df['Overall Qual'] >= quality_threshold]

    # Filter above price threshold
    high_quality_above = high_quality[high_quality['SalePrice'] > price_threshold]

    # Calculate fraction
    fraction = len(high_quality_above) / len(high_quality)

    # Print results
    print(f"High-quality items (OverallQual >= {quality_threshold}): {len(high_quality)}")
    print(f"Above ${price_threshold:,}: {len(high_quality_above)}")
    print(f"Probability (fraction): {fraction:.4f} ({fraction*100:.2f}%)")

    print(f"\n Additional Statistics ")
    print(f"Mean SalePrice of high-quality: ${high_quality['SalePrice'].mean():,.0f}")
    print(f"Median SalePrice of high-quality: ${high_quality['SalePrice'].median():,.0f}")

    return fraction

In [26]:
def clean_data(path):
    df = pd.read_csv(path)
    df = df.fillna(df.median(numeric_only=True))
    calculate_stats(df, 'SalePrice')
    standardize_feature(df, 'Gr Liv Area')
    numeric_cols = ['Gr Liv Area', 'Total Bsmt SF', 'Garage Area', 'Overall Qual', 'Year Built']
    analyze_similarity(df, numeric_cols)
    analyze_quality_price(df)
    return df

In [28]:
df_cleaned = clean_data("AmesHousing.csv")
print(df_cleaned.head())
print(df_cleaned.tail())

Manual Mean of SalePrice: 180796.0600682594
Manual Std of SalePrice: 79873.05865192253
Verification with pandas .mean(): 180796.0600682594
Verification with pandas .std(): 79886.69235666493
Manual Standardization (first 5 values): [ 0.30926506 -1.19442705 -0.33771825  1.20752324  0.25584442]
StandardScaler (first 5 values): [ 0.30926506 -1.19442705 -0.33771825  1.20752324  0.25584442]
Are they equal? True
Highest value record (index 1767):
  SalePrice: $755,000
Lowest value record (index 181):
  SalePrice: $12,789
Cosine Similarity: 0.7699
High-quality items (OverallQual >= 7): 1090
Above $200,000: 742
Probability (fraction): 0.6807 (68.07%)

 Additional Statistics 
Mean SalePrice of high-quality: $249,187
Median SalePrice of high-quality: $230,000
   Order        PID  MS SubClass MS Zoning  Lot Frontage  Lot Area Street  \
0      1  526301100           20        RL         141.0     31770   Pave   
1      2  526350040           20        RH          80.0     11622   Pave   
2      3  